##libraries and source data

In [ ]:
import requests
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, dayofmonth, concat_ws, lit, unix_timestamp, year
from pyspark.sql.types import DoubleType, TimestampType

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import shutil

source_directory = '/content/drive/MyDrive/MITBDS/parquet_files_prepared'
destination_directory = '/content/drive/MyDrive/MITBDS/parquet_files_2024_2025'

# Explicitly remove the destination directory
if os.path.exists(destination_directory):
    print(f"Removing existing destination directory: {destination_directory}")
    shutil.rmtree(destination_directory)

# Create the destination directory if it doesn't exist
os.makedirs(destination_directory, exist_ok=True)

print(f"Re-copying files from '{source_directory}' to '{destination_directory}'")

for filename in os.listdir(source_directory):
    if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
        source_path = os.path.join(source_directory, filename)
        destination_path = os.path.join(destination_directory, filename)
        try:
            shutil.copy2(source_path, destination_path)
            print(f"  Copied: {filename}")
        except Exception as e:
            print(f"  Error copying {filename}: {e}")

print("File re-copying complete. Please now re-run all subsequent cells related to impossible/outlier data processing.")

Removing existing destination directory: /content/drive/MyDrive/MITBDS/parquet_files_2024_2025
Re-copying files from '/content/drive/MyDrive/MITBDS/parquet_files_prepared' to '/content/drive/MyDrive/MITBDS/parquet_files_2024_2025'
  Copied: yellow_tripdata_2024-01.parquet
  Copied: yellow_tripdata_2024-02.parquet
  Copied: yellow_tripdata_2024-03.parquet
  Copied: yellow_tripdata_2024-04.parquet
  Copied: yellow_tripdata_2024-05.parquet
  Copied: yellow_tripdata_2024-06.parquet
  Copied: yellow_tripdata_2024-07.parquet
  Copied: yellow_tripdata_2024-08.parquet
  Copied: yellow_tripdata_2024-09.parquet
  Copied: yellow_tripdata_2024-10.parquet
  Copied: yellow_tripdata_2024-11.parquet
  Copied: yellow_tripdata_2024-12.parquet
  Copied: yellow_tripdata_2025-02.parquet
  Copied: yellow_tripdata_2025-01.parquet
  Copied: yellow_tripdata_2025-03.parquet
  Copied: yellow_tripdata_2025-04.parquet
  Copied: yellow_tripdata_2025-05.parquet
  Copied: yellow_tripdata_2025-06.parquet
  Copied: yel

In [ ]:
directory = '/content/drive/MyDrive/MITBDS/parquet_files_2024_2025'

#data preparation

### impossible durations

####identify

In [ ]:
def find_impossible_durations(spark_session, source_dir): # Renamed 'spark' parameter to 'spark_session'
    impossible_duration_summary = {}

    for filename in os.listdir(source_dir):
        if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
            file_path = os.path.join(source_dir, filename)
            print(f"Processing file: {filename}")

            try:
                df = spark_session.read.parquet(file_path) # Use passed spark_session

                pickup_col = 'tpep_pickup_datetime'
                dropoff_col = 'tpep_dropoff_datetime'

                if pickup_col in df.columns and dropoff_col in df.columns:
                    df = df.withColumn(
                        'started_at',
                        col(pickup_col).cast(TimestampType())
                    ).withColumn(
                        'ended_at',
                        col(dropoff_col).cast(TimestampType())
                    )

                    df = df.withColumn(
                        'trip_duration_seconds',
                        unix_timestamp(col('ended_at')) - unix_timestamp(col('started_at'))
                    )

                    # Trips that are less than 10 seconds long
                    impossible_trips_count = df.filter(col('trip_duration_seconds') < 10).count()

                    if impossible_trips_count > 0:
                        print(f"  Found {impossible_trips_count} trips with impossible durations.")
                        impossible_duration_summary[filename] = impossible_trips_count
                    else:
                        print("No impossible durations found in this file.")
                else:
                    print(f"  Skipping file: {filename} as '{pickup_col}' or '{dropoff_col}' column is missing.")
            except Exception as e:
                print(f"  Error processing {filename}: {e}")

    return impossible_duration_summary

# Initialize SparkSession once for the entire block
spark = SparkSession.builder.appName("ImpossibleDurations").getOrCreate()

directory = '/content/drive/MyDrive/MITBDS/parquet_files_2024_2025'
summary = find_impossible_durations(spark, directory) # Pass spark_session

print("\n--- Summary ---")
if summary:
    for file, count in summary.items():
        print(f"File: {file}, Impossible Durations: {count}")
else:
    print("No files found or no impossible durations detected.")

# Stop SparkSession
spark.stop()


Processing file: yellow_tripdata_2024-01.parquet
  Found 4797 trips with impossible durations.
Processing file: yellow_tripdata_2024-02.parquet
  Found 4556 trips with impossible durations.
Processing file: yellow_tripdata_2024-03.parquet
  Found 5530 trips with impossible durations.
Processing file: yellow_tripdata_2024-04.parquet
  Found 5141 trips with impossible durations.
Processing file: yellow_tripdata_2024-05.parquet
  Found 5460 trips with impossible durations.
Processing file: yellow_tripdata_2024-06.parquet
  Found 5625 trips with impossible durations.
Processing file: yellow_tripdata_2024-07.parquet
  Found 5391 trips with impossible durations.
Processing file: yellow_tripdata_2024-08.parquet
  Found 5691 trips with impossible durations.
Processing file: yellow_tripdata_2024-09.parquet
  Found 5958 trips with impossible durations.
Processing file: yellow_tripdata_2024-10.parquet
  Found 5934 trips with impossible durations.
Processing file: yellow_tripdata_2024-11.parquet
 

####quantify

In [ ]:
spark = SparkSession.builder.appName("QuantifyImpossibleDurations").getOrCreate()

total_trips = 0
total_impossible_durations = 0


impossible_duration_summary_spark = {}

for filename in os.listdir(directory):
    if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
        file_path = os.path.join(directory, filename)
        try:
            df = spark.read.parquet(file_path)
            total_trips += df.count()

            pickup_col = 'tpep_pickup_datetime'
            dropoff_col = 'tpep_dropoff_datetime'

            if pickup_col in df.columns and dropoff_col in df.columns:
                df = df.withColumn(
                    'started_at',
                    col(pickup_col).cast(TimestampType())
                ).withColumn(
                    'ended_at',
                    col(dropoff_col).cast(TimestampType())
                )

                df = df.withColumn(
                    'trip_duration_seconds',
                    unix_timestamp(col('ended_at')) - unix_timestamp(col('started_at'))
                )

                impossible_trips_count = df.filter(col('trip_duration_seconds') < 10).count()
                impossible_duration_summary_spark[filename] = impossible_trips_count
                total_impossible_durations += impossible_trips_count
            else:
                print(f"Skipping {filename}: Missing '{pickup_col}' or '{dropoff_col}'.")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

print(f"\nTotal trips across all 2024/2025 files: {total_trips}")
print(f"Total trips with impossible durations (< 10 seconds): {total_impossible_durations}")

if total_trips > 0:
    percentage_impossible = (total_impossible_durations / total_trips) * 100
    print(f"Percentage of trips with impossible durations: {percentage_impossible:.2f}%")
else:
    print("No trips found.")

spark.stop()


Total trips across all 2024/2025 files: 26425853
Total trips with impossible durations (< 10 seconds): 285877
Percentage of trips with impossible durations: 1.08%


####action

In [ ]:
def delete_impossible_duration_records(spark_session, source_dir):
    for filename in os.listdir(source_dir):
        if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
            original_file_path = os.path.join(source_dir, filename)
            print(f"Processing file: {filename}")

            try:
                df = spark_session.read.parquet(original_file_path)
                pickup_col = 'tpep_pickup_datetime'
                dropoff_col = 'tpep_dropoff_datetime'

                if pickup_col in df.columns and dropoff_col in df.columns:
                    # Convert to timestamp and calculate duration
                    df_with_duration = df.withColumn(
                        'started_at',
                        col(pickup_col).cast(TimestampType())
                    ).withColumn(
                        'ended_at',
                        col(dropoff_col).cast(TimestampType())
                    ).withColumn(
                        'trip_duration_seconds',
                        unix_timestamp(col('ended_at')) - unix_timestamp(col('started_at'))
                    )

                    # Identify records to keep (duration >= 10 seconds)
                    df_cleaned = df_with_duration.filter(col('trip_duration_seconds') >= 10)

                    num_original_records = df.count()
                    num_cleaned_records = df_cleaned.count()
                    deleted_records_count = num_original_records - num_cleaned_records

                    if deleted_records_count > 0:
                        print(f"  Deleted {deleted_records_count} records with impossible durations (duration < 10 seconds).")

                        # --- MODIFIED OVERWRITE LOGIC START ---
                        temp_output_dir = os.path.join(source_dir, f"temp_spark_output_{os.urandom(4).hex()}")

                        # Write the cleaned DataFrame to a temporary directory as a single Parquet file
                        df_cleaned.coalesce(1).write.mode("overwrite").parquet(temp_output_dir)

                        # Get the actual part-file name from the temporary directory
                        part_files = [f for f in os.listdir(temp_output_dir) if f.startswith('part-') and f.endswith('.parquet')]
                        if not part_files:
                            raise Exception(f"No part files found in temporary directory {temp_output_dir}")
                        temp_part_file_path = os.path.join(temp_output_dir, part_files[0])

                        # Delete the original file/directory if it exists
                        if os.path.exists(original_file_path):
                            if os.path.isdir(original_file_path):
                                shutil.rmtree(original_file_path)
                            else:
                                os.remove(original_file_path)

                        # Move the single part-file to the original file's location
                        shutil.move(temp_part_file_path, original_file_path)
                        shutil.rmtree(temp_output_dir)

                        print(f"  Updated file saved: {filename}")
                    else:
                        print("  No records found to delete.")
                else:
                    print(f"  Skipping file: {filename} as '{pickup_col}' or '{dropoff_col}' column is missing.")
            except Exception as e:
                print(f"  Error processing {filename}: {e}")

    print("Done.")

# Initialize SparkSession once for the entire cell
spark = SparkSession.builder.appName("DeleteImpossibleDurations").getOrCreate()
delete_impossible_duration_records(spark, directory)

# Stop SparkSession
spark.stop()


Processing file: yellow_tripdata_2024-01.parquet
  Deleted 4797 records with impossible durations (duration < 10 seconds).
  Updated file saved: yellow_tripdata_2024-01.parquet
Processing file: yellow_tripdata_2024-02.parquet
  Deleted 4556 records with impossible durations (duration < 10 seconds).
  Updated file saved: yellow_tripdata_2024-02.parquet
Processing file: yellow_tripdata_2024-03.parquet
  Deleted 5530 records with impossible durations (duration < 10 seconds).
  Updated file saved: yellow_tripdata_2024-03.parquet
Processing file: yellow_tripdata_2024-04.parquet
  Deleted 5141 records with impossible durations (duration < 10 seconds).
  Updated file saved: yellow_tripdata_2024-04.parquet
Processing file: yellow_tripdata_2024-05.parquet
  Deleted 5460 records with impossible durations (duration < 10 seconds).
  Updated file saved: yellow_tripdata_2024-05.parquet
Processing file: yellow_tripdata_2024-06.parquet
  Deleted 5625 records with impossible durations (duration < 10 se

### impossible distances

####identify

In [ ]:
def find_impossible_distances(spark, source_dir):
    impossible_distance_summary = {}

    for filename in os.listdir(source_dir):
        if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
            file_path = os.path.join(source_dir, filename)
            print(f"Processing file: {filename}")

            try:
                df = spark.read.parquet(file_path)
                distance_col = 'trip_distance'

                if distance_col in df.columns:

                    impossible_distance_trips_count = df.filter(col(distance_col) <= 0).count()

                    if impossible_distance_trips_count > 0:
                        print(f"  Found {impossible_distance_trips_count} trips with impossible distances (<= 0).")
                        impossible_distance_summary[filename] = impossible_distance_trips_count
                    else:
                        print("  No impossible distances found in this file.")
                else:
                    print(f"  Skipping file: {filename} as '{distance_col}' column is missing.")
            except Exception as e:
                print(f"  Error processing {filename}: {e}")

    return impossible_distance_summary

# Initialize SparkSession if not already initialized (for standalone execution)
spark = SparkSession.builder.appName("ImpossibleDistances").getOrCreate()

distance_summary = find_impossible_distances(spark, directory)

print("\n--- Impossible Distances Summary ---")
if distance_summary:
    for file, count in distance_summary.items():
        print(f"File: {file}, Impossible Distances: {count}")
else:
    print("No files found or no impossible distances detected.")


Processing file: yellow_tripdata_2024-01.parquet
  Found 16814 trips with impossible distances (<= 0).
Processing file: yellow_tripdata_2024-02.parquet
  Found 21905 trips with impossible distances (<= 0).
Processing file: yellow_tripdata_2024-03.parquet
  Found 27679 trips with impossible distances (<= 0).
Processing file: yellow_tripdata_2024-04.parquet
  Found 9001 trips with impossible distances (<= 0).
Processing file: yellow_tripdata_2024-05.parquet
  Found 9095 trips with impossible distances (<= 0).
Processing file: yellow_tripdata_2024-06.parquet
  Found 9713 trips with impossible distances (<= 0).
Processing file: yellow_tripdata_2024-07.parquet
  Found 8507 trips with impossible distances (<= 0).
Processing file: yellow_tripdata_2024-08.parquet
  Found 8410 trips with impossible distances (<= 0).
Processing file: yellow_tripdata_2024-09.parquet
  Found 9988 trips with impossible distances (<= 0).
Processing file: yellow_tripdata_2024-10.parquet
  Found 12193 trips with impos

####quantify

In [ ]:
spark = SparkSession.builder.appName("QuantifyImpossibleDistances").getOrCreate()

total_trips_distance = 0
total_impossible_distances = 0


for filename in os.listdir(directory):
    if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
        file_path = os.path.join(directory, filename)
        try:
            df = spark.read.parquet(file_path)
            total_trips_distance += df.count()

            distance_col = 'trip_distance'
            if distance_col in df.columns:
                impossible_trips_count_file = df.filter(col(distance_col) <= 0).count()
                total_impossible_distances += impossible_trips_count_file
            else:
                print(f"Skipping {filename}: Missing '{distance_col}'.")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

print(f"\nTotal trips across all 2024/2025 files (for distance analysis): {total_trips_distance}")
print(f"Total trips with impossible distances (<= 0): {total_impossible_distances}")

if total_trips_distance > 0:
    percentage_impossible = (total_impossible_distances / total_trips_distance) * 100
    print(f"Percentage of trips with impossible distances: {percentage_impossible:.2f}%")
else:
    print("No trips found to calculate percentage.")

# Stop SparkSession
spark.stop()


Total trips across all 2024/2025 files (for distance analysis): 26139976
Total trips with impossible distances (<= 0): 265582
Percentage of trips with impossible distances: 1.02%


####action

In [ ]:
def delete_impossible_distance_records(spark_session, source_dir):
    for filename in os.listdir(source_dir):
        if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
            original_file_path = os.path.join(source_dir, filename)
            print(f"Processing file: {filename}")

            try:
                df = spark_session.read.parquet(original_file_path)
                distance_col = 'trip_distance'

                if distance_col in df.columns:
                    # Records to keep (distance > 0)
                    df_cleaned = df.filter(col(distance_col) > 0)

                    num_original_records = df.count()
                    num_cleaned_records = df_cleaned.count()
                    deleted_records_count = num_original_records - num_cleaned_records

                    if deleted_records_count > 0:
                        print(f"  Deleted {deleted_records_count} records with impossible distances (distance <= 0).")

                        # --- MODIFIED OVERWRITE LOGIC START ---
                        temp_output_dir = os.path.join(source_dir, f"temp_spark_output_{os.urandom(4).hex()}")

                        # Write the cleaned DataFrame to a temporary directory as a single Parquet file
                        df_cleaned.coalesce(1).write.mode("overwrite").parquet(temp_output_dir)

                        # Get the actual part-file name from the temporary directory
                        part_files = [f for f in os.listdir(temp_output_dir) if f.startswith('part-') and f.endswith('.parquet')]
                        if not part_files:
                            raise Exception(f"No part files found in temporary directory {temp_output_dir}")
                        temp_part_file_path = os.path.join(temp_output_dir, part_files[0])

                        # Delete the original file/directory if it exists
                        if os.path.exists(original_file_path):
                            if os.path.isdir(original_file_path):
                                shutil.rmtree(original_file_path)
                            else:
                                os.remove(original_file_path)

                        # Move the single part-file to the original file's location
                        shutil.move(temp_part_file_path, original_file_path)
                        shutil.rmtree(temp_output_dir)

                        print(f"  Updated file saved: {filename}")
                    else:
                        print("  No records found to delete.")
                else:
                    print(f"  Skipping file: {filename} as '{distance_col}' column is missing.")
            except Exception as e:
                print(f"  Error processing {filename}: {e}")

    print("Done.")

# Initialize SparkSession for standalone execution of this cell if needed
spark = SparkSession.builder.appName("DeleteImpossibleDistances").getOrCreate()

delete_impossible_distance_records(spark, directory)

# Stop SparkSession
spark.stop()

Processing file: yellow_tripdata_2024-01.parquet
  No records found to delete.
Processing file: yellow_tripdata_2024-02.parquet
  No records found to delete.
Processing file: yellow_tripdata_2024-03.parquet
  No records found to delete.
Processing file: yellow_tripdata_2024-04.parquet
  No records found to delete.
Processing file: yellow_tripdata_2024-05.parquet
  No records found to delete.
Processing file: yellow_tripdata_2024-06.parquet
  No records found to delete.
Processing file: yellow_tripdata_2024-07.parquet
  No records found to delete.
Processing file: yellow_tripdata_2024-08.parquet
  No records found to delete.
Processing file: yellow_tripdata_2024-09.parquet
  No records found to delete.
Processing file: yellow_tripdata_2024-10.parquet
  No records found to delete.
Processing file: yellow_tripdata_2024-11.parquet
  No records found to delete.
Processing file: yellow_tripdata_2024-12.parquet
  No records found to delete.
Processing file: yellow_tripdata_2025-02.parquet
  N

###impossible speeds

####identify

In [ ]:
from pyspark.sql.functions import col, unix_timestamp, lit, when

def find_impossible_speeds(spark, source_dir):
    impossible_speed_summary = {}

    MAX_REALISTIC_SPEED_MPH = 100

    for filename in os.listdir(source_dir):
        if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
            file_path = os.path.join(source_dir, filename)
            print(f"Processing file: {filename}")

            try:
                df = spark.read.parquet(file_path)

                pickup_col = 'tpep_pickup_datetime'
                dropoff_col = 'tpep_dropoff_datetime'
                distance_col = 'trip_distance'

                if pickup_col in df.columns and dropoff_col in df.columns and distance_col in df.columns:
                    df = df.withColumn(
                        'started_at',
                        col(pickup_col).cast(TimestampType())
                    ).withColumn(
                        'ended_at',
                        col(dropoff_col).cast(TimestampType())
                    ).withColumn(
                        'trip_duration_seconds',
                        unix_timestamp(col('ended_at')) - unix_timestamp(col('started_at'))
                    )

                    # Calculate avg_speed_mph, handling division by zero/nulls
                    df = df.withColumn(
                        'avg_speed_mph',
                        when(
                            (col('trip_duration_seconds') > 0) & (col(distance_col) > 0),
                            col(distance_col) / (col('trip_duration_seconds') / 3600)
                        ).otherwise(lit(None))
                    )

                    # Define "impossible speed" conditions:
                    # Duration <= 0 or is null
                    # Distance <= 0 or is null
                    # Speed > MAX_REALISTIC_SPEED_MPH or speed is null (which would include cases where duration/distance are problematic)
                    impossible_speeds_condition = (
                        (col('trip_duration_seconds') <= 0) |
                        (col('trip_duration_seconds').isNull()) |
                        (col(distance_col) <= 0) |
                        (col(distance_col).isNull()) |
                        (col('avg_speed_mph').isNull()) | # Includes cases where speed couldn't be calculated meaningfully
                        (col('avg_speed_mph') > MAX_REALISTIC_SPEED_MPH)
                    )

                    impossible_trips_count = df.filter(impossible_speeds_condition).count()

                    if impossible_trips_count > 0:
                        print(f"  Found {impossible_trips_count} trips with impossible speeds (e.g., duration <=0, distance <=0, or speed > {MAX_REALISTIC_SPEED_MPH} mph).")
                        impossible_speed_summary[filename] = impossible_trips_count
                    else:
                        print("  No impossible speeds found in this file.")
                else:
                    missing_cols = []
                    if pickup_col not in df.columns: missing_cols.append(pickup_col)
                    if dropoff_col not in df.columns: missing_cols.append(dropoff_col)
                    if distance_col not in df.columns: missing_cols.append(distance_col)
                    print(f"  Skipping file: {filename} due to missing columns: {', '.join(missing_cols)}.")
            except Exception as e:
                print(f"  Error processing {filename}: {e}")

    return impossible_speed_summary

# Initialize SparkSession for standalone execution of this cell if needed
spark = SparkSession.builder.appName("ImpossibleSpeeds").getOrCreate()

speed_summary = find_impossible_speeds(spark, directory)

print("\n--- Summary ---")
if speed_summary:
    for file, count in speed_summary.items():
        print(f"File: {file}, Impossible Speeds: {count}")
else:
    print("No files found or no impossible speeds detected.")

# Stop SparkSession
spark.stop()

Processing file: yellow_tripdata_2024-01.parquet
  Found 177 trips with impossible speeds (e.g., duration <=0, distance <=0, or speed > 100 mph).
Processing file: yellow_tripdata_2024-02.parquet
  Found 152 trips with impossible speeds (e.g., duration <=0, distance <=0, or speed > 100 mph).
Processing file: yellow_tripdata_2024-03.parquet
  Found 207 trips with impossible speeds (e.g., duration <=0, distance <=0, or speed > 100 mph).
Processing file: yellow_tripdata_2024-04.parquet
  Found 210 trips with impossible speeds (e.g., duration <=0, distance <=0, or speed > 100 mph).
Processing file: yellow_tripdata_2024-05.parquet
  Found 216 trips with impossible speeds (e.g., duration <=0, distance <=0, or speed > 100 mph).
Processing file: yellow_tripdata_2024-06.parquet
  Found 226 trips with impossible speeds (e.g., duration <=0, distance <=0, or speed > 100 mph).
Processing file: yellow_tripdata_2024-07.parquet
  Found 246 trips with impossible speeds (e.g., duration <=0, distance <=0,

####quantify

In [ ]:
spark = SparkSession.builder.appName("QuantifyImpossibleSpeeds").getOrCreate()

total_trips_speed = 0
total_impossible_speeds = 0

for filename in os.listdir(directory):
    if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
        file_path = os.path.join(directory, filename)
        try:
            df = spark.read.parquet(file_path)
            total_trips_speed += df.count()

            pickup_col = 'tpep_pickup_datetime'
            dropoff_col = 'tpep_dropoff_datetime'
            distance_col = 'trip_distance'
            MAX_REALISTIC_SPEED_MPH = 100

            if pickup_col in df.columns and dropoff_col in df.columns and distance_col in df.columns:
                df_with_duration = df.withColumn(
                    'started_at',
                    col(pickup_col).cast(TimestampType())
                ).withColumn(
                    'ended_at',
                    col(dropoff_col).cast(TimestampType())
                ).withColumn(
                    'trip_duration_seconds',
                    unix_timestamp(col('ended_at')) - unix_timestamp(col('started_at'))
                )

                df_with_speed = df_with_duration.withColumn(
                    'avg_speed_mph',
                    when(
                        (col('trip_duration_seconds') > 0) & (col(distance_col) > 0),
                        col(distance_col) / (col('trip_duration_seconds') / 3600)
                    ).otherwise(lit(None))
                )

                impossible_speeds_condition = (
                    (col('trip_duration_seconds') <= 0) |
                    (col('trip_duration_seconds').isNull()) |
                    (col(distance_col) <= 0) |
                    (col(distance_col).isNull()) |
                    (col('avg_speed_mph').isNull()) |
                    (col('avg_speed_mph') > MAX_REALISTIC_SPEED_MPH)
                )

                impossible_trips_count_file = df_with_speed.filter(impossible_speeds_condition).count()
                total_impossible_speeds += impossible_trips_count_file
            else:
                print(f"Skipping {filename}: Missing '{pickup_col}', '{dropoff_col}', or '{distance_col}'.")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

print(f"\nTotal trips across all 2024/2025 files: {total_trips_speed}")
print(f"Total trips with impossible speeds (e.g., duration <=0, distance <=0, or speed > 100 mph): {total_impossible_speeds}")

if total_trips_speed > 0:
    percentage_impossible_speeds = (total_impossible_speeds / total_trips_speed) * 100
    print(f"Percentage of trips with impossible speeds: {percentage_impossible_speeds:.2f}%")
else:
    print("No trips found to calculate percentage.")

# Stop SparkSession
spark.stop()


Total trips across all 2024/2025 files: 25874394
Total trips with impossible speeds (e.g., duration <=0, distance <=0, or speed > 100 mph): 4579
Percentage of trips with impossible speeds: 0.02%


#### action

In [ ]:
from pyspark.sql.functions import col, unix_timestamp, lit, when

def delete_impossible_speed_records(spark_session, source_dir):
    MAX_REALISTIC_SPEED_MPH = 100

    for filename in os.listdir(source_dir):
        if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
            original_file_path = os.path.join(source_dir, filename)
            print(f"Processing file: {filename}")

            try:
                df = spark_session.read.parquet(original_file_path)

                pickup_col = 'tpep_pickup_datetime'
                dropoff_col = 'tpep_dropoff_datetime'
                distance_col = 'trip_distance'

                if pickup_col in df.columns and dropoff_col in df.columns and distance_col in df.columns:
                    df_with_calculations = df.withColumn(
                        'started_at',
                        col(pickup_col).cast(TimestampType())
                    ).withColumn(
                        'ended_at',
                        col(dropoff_col).cast(TimestampType())
                    ).withColumn(
                        'trip_duration_seconds',
                        unix_timestamp(col('ended_at')) - unix_timestamp(col('started_at'))
                    ).withColumn(
                        'avg_speed_mph',
                        when(
                            (col('trip_duration_seconds') > 0) & (col(distance_col) > 0),
                            col(distance_col) / (col('trip_duration_seconds') / 3600)
                        ).otherwise(lit(None))
                    )

                    # Define "impossible speed" conditions:
                    impossible_speeds_condition = (
                        (col('trip_duration_seconds') <= 0) |
                        (col('trip_duration_seconds').isNull()) |
                        (col(distance_col) <= 0) |
                        (col(distance_col).isNull()) |
                        (col('avg_speed_mph').isNull()) | # Includes cases where speed couldn't be calculated meaningfully
                        (col('avg_speed_mph') > MAX_REALISTIC_SPEED_MPH)
                    )

                    # Filter out records with impossible speeds
                    df_cleaned = df_with_calculations.filter(~impossible_speeds_condition)

                    num_original_records = df.count()
                    num_cleaned_records = df_cleaned.count()
                    deleted_records_count = num_original_records - num_cleaned_records

                    if deleted_records_count > 0:
                        print(f"  Deleted {deleted_records_count} records with impossible speeds.")

                        # --- MODIFIED OVERWRITE LOGIC START ---
                        temp_output_dir = os.path.join(source_dir, f"temp_spark_output_{os.urandom(4).hex()}")
                        df_cleaned.coalesce(1).write.mode("overwrite").parquet(temp_output_dir)

                        part_files = [f for f in os.listdir(temp_output_dir) if f.startswith('part-') and f.endswith('.parquet')]
                        if not part_files:
                            raise Exception(f"No part files found in temporary directory {temp_output_dir}")
                        temp_part_file_path = os.path.join(temp_output_dir, part_files[0])

                        if os.path.exists(original_file_path):
                            if os.path.isdir(original_file_path):
                                shutil.rmtree(original_file_path)
                            else:
                                os.remove(original_file_path)

                        shutil.move(temp_part_file_path, original_file_path)
                        shutil.rmtree(temp_output_dir)
                        # --- MODIFIED OVERWRITE LOGIC END ---

                        print(f"  Updated file saved: {filename}")
                    else:
                        print("  No records found to delete.")
                else:
                    missing_cols = []
                    if pickup_col not in df.columns: missing_cols.append(pickup_col)
                    if dropoff_col not in df.columns: missing_cols.append(dropoff_col)
                    if distance_col not in df.columns: missing_cols.append(distance_col)
                    print(f"  Skipping file: {filename} due to missing columns: {', '.join(missing_cols)}.")
            except Exception as e:
                print(f"  Error processing {filename}: {e}")

    print("Done.")

# Initialize SparkSession for standalone execution of this cell if needed
spark = SparkSession.builder.appName("DeleteImpossibleSpeeds").getOrCreate()

delete_impossible_speed_records(spark, directory)

# Stop SparkSession
spark.stop()


Processing file: yellow_tripdata_2024-01.parquet
  Deleted 177 records with impossible speeds.
  Updated file saved: yellow_tripdata_2024-01.parquet
Processing file: yellow_tripdata_2024-02.parquet
  Deleted 152 records with impossible speeds.
  Updated file saved: yellow_tripdata_2024-02.parquet
Processing file: yellow_tripdata_2024-03.parquet
  Deleted 207 records with impossible speeds.
  Updated file saved: yellow_tripdata_2024-03.parquet
Processing file: yellow_tripdata_2024-04.parquet
  Deleted 210 records with impossible speeds.
  Updated file saved: yellow_tripdata_2024-04.parquet
Processing file: yellow_tripdata_2024-05.parquet
  Deleted 216 records with impossible speeds.
  Updated file saved: yellow_tripdata_2024-05.parquet
Processing file: yellow_tripdata_2024-06.parquet
  Deleted 226 records with impossible speeds.
  Updated file saved: yellow_tripdata_2024-06.parquet
Processing file: yellow_tripdata_2024-07.parquet
  Deleted 246 records with impossible speeds.
  Updated f

##outlier distances

###identify

In [ ]:
from pyspark.sql.functions import col, unix_timestamp, percent_rank, count, lit
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, DoubleType

spark = SparkSession.builder.appName("OutlierAnalysis").getOrCreate()

def find_outlier_distances(spark, source_dir, percentile_threshold=0.99):
    outlier_distance_summary = {}

    for filename in os.listdir(source_dir):
        if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
            file_path = os.path.join(source_dir, filename)
            print(f"Processing file: {filename}")

            try:
                df = spark.read.parquet(file_path)
                distance_col = 'trip_distance'

                if distance_col in df.columns:
                    # Calculate 99th percentile for trip_distance
                    # Using approx_quantile for efficiency on large datasets
                    quantiles = df.approxQuantile(distance_col, [percentile_threshold], 0.01)
                    if quantiles and len(quantiles) > 0:
                        upper_bound_distance = quantiles[0]
                    else:
                        print(f"  Could not calculate quantile for {filename}. Skipping outlier check.")
                        continue

                    # Filter for outlier distances (above 99th percentile and > 0)
                    outlier_trips_count = df.filter(
                        (col(distance_col) > upper_bound_distance) & (col(distance_col) > 0)
                    ).count()

                    if outlier_trips_count > 0:
                        print(f"  Found {outlier_trips_count} trips with outlier distances (>{upper_bound_distance:.2f} miles).")
                        outlier_distance_summary[filename] = outlier_trips_count
                    else:
                        print("  No outlier distances found in this file.")
                else:
                    print(f"  Skipping file: {filename} as '{distance_col}' column is missing.")
            except Exception as e:
                print(f"  Error processing {filename}: {e}")

    return outlier_distance_summary

# Execute Identify Outlier Distances
outlier_dist_summary = find_outlier_distances(spark, directory)

print("\n--- Identified Outlier Distances Summary ---")
if outlier_dist_summary:
    for file, count_val in outlier_dist_summary.items():
        print(f"File: {file}, Outlier Distances: {count_val}")
else:
    print("No files found or no outlier distances detected.")


Processing file: yellow_tripdata_2024-01.parquet
  No outlier distances found in this file.
Processing file: yellow_tripdata_2024-02.parquet
  No outlier distances found in this file.
Processing file: yellow_tripdata_2024-03.parquet
  No outlier distances found in this file.
Processing file: yellow_tripdata_2024-04.parquet
  No outlier distances found in this file.
Processing file: yellow_tripdata_2024-05.parquet
  No outlier distances found in this file.
Processing file: yellow_tripdata_2024-06.parquet
  No outlier distances found in this file.
Processing file: yellow_tripdata_2024-07.parquet
  No outlier distances found in this file.
Processing file: yellow_tripdata_2024-08.parquet
  No outlier distances found in this file.
Processing file: yellow_tripdata_2024-09.parquet
  No outlier distances found in this file.
Processing file: yellow_tripdata_2024-10.parquet
  No outlier distances found in this file.
Processing file: yellow_tripdata_2024-11.parquet
  No outlier distances found in

### quantify

In [ ]:
total_trips_dist_quantify = 0
total_outlier_distances_quantify = 0

for filename in os.listdir(directory):
    if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
        file_path = os.path.join(directory, filename)
        try:
            df = spark.read.parquet(file_path)
            total_trips_dist_quantify += df.count()

            distance_col = 'trip_distance'
            if distance_col in df.columns:
                quantiles = df.approxQuantile(distance_col, [0.99], 0.01)
                if quantiles and len(quantiles) > 0:
                    upper_bound_distance = quantiles[0]
                else:
                    upper_bound_distance = float('inf') # If no quantile, no upper bound, assume no outliers for this file

                outlier_trips_count_file = df.filter(
                    (col(distance_col) > upper_bound_distance) & (col(distance_col) > 0)
                ).count()
                total_outlier_distances_quantify += outlier_trips_count_file
            else:
                print(f"Skipping {filename}: Missing '{distance_col}'.")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

print(f"Total trips across all 2024/2025 files (for distance outlier analysis): {total_trips_dist_quantify}")
print(f"Total trips with outlier distances: {total_outlier_distances_quantify}")

if total_trips_dist_quantify > 0:
    percentage_outlier_distances = (total_outlier_distances_quantify / total_trips_dist_quantify) * 100
    print(f"Percentage of trips with outlier distances: {percentage_outlier_distances:.2f}%")
else:
    print("No trips found to calculate percentage for outlier distances.")

Total trips across all 2024/2025 files (for distance outlier analysis): 25869815
Total trips with outlier distances: 0
Percentage of trips with outlier distances: 0.00%


### action

*   No action required



##outlier durations

### identify

In [ ]:
def find_outlier_durations(spark, source_dir, percentile_threshold=0.99):
    outlier_duration_summary = {}

    for filename in os.listdir(source_dir):
        if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
            file_path = os.path.join(source_dir, filename)
            print(f"Processing file: {filename}")

            try:
                df = spark.read.parquet(file_path)

                pickup_col = 'tpep_pickup_datetime'
                dropoff_col = 'tpep_dropoff_datetime'

                if pickup_col in df.columns and dropoff_col in df.columns:
                    df_with_duration = df.withColumn(
                        'started_at',
                        col(pickup_col).cast(TimestampType())
                    ).withColumn(
                        'ended_at',
                        col(dropoff_col).cast(TimestampType())
                    ).withColumn(
                        'trip_duration_seconds',
                        (unix_timestamp(col('ended_at')) - unix_timestamp(col('started_at'))).cast(DoubleType())
                    )

                    # Calculate 99th percentile for trip_duration_seconds
                    quantiles = df_with_duration.approxQuantile('trip_duration_seconds', [percentile_threshold], 0.01)
                    if quantiles and len(quantiles) > 0:
                        upper_bound_duration = quantiles[0]
                    else:
                        print(f"  Could not calculate quantile for {filename}. Skipping outlier check.")
                        continue

                    # Filter for outlier durations (above 99th percentile and > 0)
                    outlier_trips_count = df_with_duration.filter(
                        (col('trip_duration_seconds') > upper_bound_duration) & (col('trip_duration_seconds') > 0)
                    ).count()

                    if outlier_trips_count > 0:
                        print(f"  Found {outlier_trips_count} trips with outlier durations (>{upper_bound_duration:.2f} seconds).")
                        outlier_duration_summary[filename] = outlier_trips_count
                    else:
                        print("  No outlier durations found in this file.")
                else:
                    print(f"  Skipping file: {filename} as '{pickup_col}' or '{dropoff_col}' column is missing.")
            except Exception as e:
                print(f"  Error processing {filename}: {e}")

    return outlier_duration_summary

# Execute Identify Outlier Durations
outlier_dur_summary = find_outlier_durations(spark, directory)

print("\n--- Identified Outlier Durations Summary ---")
if outlier_dur_summary:
    for file, count_val in outlier_dur_summary.items():
        print(f"File: {file}, Outlier Durations: {count_val}")
else:
    print("No files found or no outlier durations detected.")


Processing file: yellow_tripdata_2024-01.parquet
  No outlier durations found in this file.
Processing file: yellow_tripdata_2024-02.parquet
  No outlier durations found in this file.
Processing file: yellow_tripdata_2024-03.parquet
  No outlier durations found in this file.
Processing file: yellow_tripdata_2024-04.parquet
  No outlier durations found in this file.
Processing file: yellow_tripdata_2024-05.parquet
  No outlier durations found in this file.
Processing file: yellow_tripdata_2024-06.parquet
  No outlier durations found in this file.
Processing file: yellow_tripdata_2024-07.parquet
  No outlier durations found in this file.
Processing file: yellow_tripdata_2024-08.parquet
  No outlier durations found in this file.
Processing file: yellow_tripdata_2024-09.parquet
  No outlier durations found in this file.
Processing file: yellow_tripdata_2024-10.parquet
  No outlier durations found in this file.
Processing file: yellow_tripdata_2024-11.parquet
  No outlier durations found in

###quantify

In [ ]:
total_trips_dur_quantify = 0
total_outlier_durations_quantify = 0

for filename in os.listdir(directory):
    if ('2024-' in filename or '2025-' in filename) and filename.endswith('.parquet'):
        file_path = os.path.join(directory, filename)
        try:
            df = spark.read.parquet(file_path)
            total_trips_dur_quantify += df.count()

            pickup_col = 'tpep_pickup_datetime'
            dropoff_col = 'tpep_dropoff_datetime'

            if pickup_col in df.columns and dropoff_col in df.columns:
                df_with_duration = df.withColumn(
                    'started_at',
                    col(pickup_col).cast(TimestampType())
                ).withColumn(
                    'ended_at',
                    col(dropoff_col).cast(TimestampType())
                ).withColumn(
                    'trip_duration_seconds',
                    (unix_timestamp(col('ended_at')) - unix_timestamp(col('started_at'))).cast(DoubleType())
                )

                quantiles = df_with_duration.approxQuantile('trip_duration_seconds', [0.99], 0.01)
                if quantiles and len(quantiles) > 0:
                    upper_bound_duration = quantiles[0]
                else:
                    upper_bound_duration = float('inf')

                outlier_trips_count_file = df_with_duration.filter(
                    (col('trip_duration_seconds') > upper_bound_duration) & (col('trip_duration_seconds') > 0)
                ).count()
                total_outlier_durations_quantify += outlier_trips_count_file
            else:
                print(f"Skipping {filename}: Missing '{pickup_col}' or '{dropoff_col}'.")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

print(f"Total trips across all 2024/2025 files (for duration outlier analysis): {total_trips_dur_quantify}")
print(f"Total trips with outlier durations: {total_outlier_durations_quantify}")

if total_trips_dur_quantify > 0:
    percentage_outlier_durations = (total_outlier_durations_quantify / total_trips_dur_quantify) * 100
    print(f"Percentage of trips with outlier durations: {percentage_outlier_durations:.2f}%")
else:
    print("No trips found to calculate percentage for outlier durations.")


Total trips across all 2024/2025 files (for duration outlier analysis): 25869815
Total trips with outlier durations: 0
Percentage of trips with outlier durations: 0.00%


### action


*   No action required

